# LangChain 101

A comprehensive, hands-on introduction to **LangChain** for beginners. This tutorial covers core concepts and provides small demos you can easily remix.

**What you'll learn**
- Install & set up on Colab
- Call an LLM (OpenAI) the "LangChain way"
- Prompts & templates
- Output parsing (get clean strings, JSON)
- Runnables & Chains (compose steps like LEGO)
- Document loading & chunking
- Tiny RAG demo with **FAISS** (local vector store)

## Install packages

In [ ]:
%pip -q install -U "langchain>=0.3.0" langchain-openai langchain-community
%pip -q install -U faiss-cpu tiktoken pypdf python-dotenv pydantic


## 1) Provide your API key


In [ ]:
import os
from google.colab import userdata

# Retrieve the API key from Colab secrets
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

# Check if the API key is set
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found in Colab secrets")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

print("OpenAI API Key is present.")

## 2) Hello, LLM (LangChain client)

Let's start by making a simple call to the OpenAI LLM using LangChain.

In [ ]:
from langchain_openai import ChatOpenAI

# Initialize the ChatOpenAI model
# We'll use a smaller model for this tutorial to keep things fast and cost-effective
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

# Invoke the LLM with a simple prompt
response = llm.invoke("In one sentence, what is LangChain?")

# Print the content of the response
print(response.content)

## 3) Prompting with templates

Prompt templates help manage and format prompts. They allow you to easily insert variables into your prompts.

### How LangChain Structures Prompts

LangChain provides tools to manage and structure prompts effectively, especially for chat models. The core concept is using `PromptTemplates`.

- **PromptTemplates:** These are objects that help create prompts from a combination of template strings and a set of parameters. Instead of hardcoding prompts, you can define templates with placeholders for dynamic values.
- **ChatPromptTemplates:** Specifically for chat models, these templates are structured as a list of messages, each associated with a role (like "system", "human", or "ai"). This aligns with how chat models are trained and helps guide the model's response based on the context provided by different speakers.
- **Message Roles:**
    - **System:** Provides initial instructions or context to the AI, setting its behavior or persona.
    - **Human:** Represents input from the user.
    - **AI:** Represents responses from the AI (useful for providing few-shot examples).
- **Partial PromptTemplates:** You can 'partially' fill a prompt template with some variables, leaving others open to be filled later. This is useful for adding fixed instructions (like output format) to a template that also takes user input.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, AIMessage

# Define a prompt template with few-shot examples
explain_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful, concise tutor who explains concepts simply and provides analogies."),

    # Few-shot example 1
    HumanMessage(content="Explain machine learning in simple terms and give one analogy."),
    AIMessage(content="Machine learning is a way to teach computers to learn from data without being explicitly programmed. They find patterns and make predictions. Analogy: Like teaching a child to identify animals by showing them pictures; they learn to recognize features on their own."),
    # Few-shot example 2
    HumanMessage(content="Explain blockchain in simple terms and give one analogy."),
    AIMessage(content="Blockchain is a digital, decentralized ledger that records transactions across many computers so that the record cannot be altered retroactively without the alteration of all subsequent blocks and the consensus of the network. Analogy: Think of it like a shared Google Sheet that everyone can see and verify, but no single person can change past entries without everyone agreeing."),

    # The actual prompt for the LLM
    ("human", "Explain {topic} in simple terms and give one analogy."),
])

# Create a chain with the prompt template and the LLM
chain = explain_prompt | llm

# Invoke the chain with a topic
response = chain.invoke({"topic":"embeddings"})

# Print the response content
print(response.content)

# Invoke the chain with another topic to show it still works
response_2 = chain.invoke({"topic":"reinforcement learning"})
print("\nAnother Example")
print(response_2.content)

## 4) Output parsing (string / JSON)

Output parsers help structure the output from the LLM into a desired format, like a simple string or JSON.

### String Output Parser

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# Initialize a string output parser
parser = StrOutputParser()

# Create a chain with the prompt, LLM, and string parser
string_chain = explain_prompt | llm | parser

# Invoke the chain
response = string_chain.invoke({"topic":"vector databases"})

# Print the parsed output
print(response)

### JSON Output Parser

You can define a schema for the desired JSON output using Pydantic.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

# Initialize a JSON output parser
json_parser = JsonOutputParser()

# Define a prompt template that asks the LLM to format output as JSON
# Define the desired JSON structure directly in the prompt instructions
json_prompt = ChatPromptTemplate.from_messages([
    ("system", "Format output as JSON."),
    ("human", """Create a mini fact card about {topic} with 2-3 bullets.
Desired JSON format:
{{
  "title": "short title",
  "bullets": ["bullet point 1", "bullet point 2", ...]
}}
""")
])

# Create a chain with the prompt, LLM, and JSON parser
json_chain = json_prompt | llm | json_parser

# Invoke the chain
out = json_chain.invoke({"topic":"cosine similarity"})
out_2 = json_chain.invoke({"topic":"UVA McIntire"})

# Display the parsed JSON output
display(out)
print("\nAnother Example:\n")
display(out_2)

## 5) Runnables (compose steps)

Runnables are the fundamental building blocks in LangChain that allow you to create sequences and chains of operations. Think of them as individual components that can be easily linked together.

You can compose Runnables using the `|` operator (pronounced "pipe"). This operator takes the output of the Runnable on the left and passes it as the input to the Runnable on the right. This makes it easy to build complex workflows by chaining together simpler steps, like passing user input through a prompt template, then to an LLM, and finally through an output parser.

Key concepts:
- **Modular Components:** Each Runnable performs a specific task (e.g., formatting a prompt, calling an LLM, parsing output).
- **Composition with `|`:** The pipe operator allows you to connect Runnables sequentially.
- **Input/Output:** Each Runnable takes an input and produces an output, which can then be used as the input for the next Runnable in the chain.
- **`RunnableLambda`:** This is a special type of Runnable that allows you to wrap any standard Python function and use it within a chain. This is incredibly powerful for adding custom processing steps to your LangChain workflows.

In [ ]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# Define simple functions to use as Runnables
def to_upper(x):
    """Converts the 'text' field to uppercase."""
    return {"text": x["text"].upper()}

def add_suffix(x):
    """Adds a suffix to the input text."""
    return x["text"] + "!!!!"

# Compose Runnables into a pipeline
pipe = (RunnablePassthrough() # Passes the input through
        | RunnableLambda(to_upper) # Applies the to_upper function
        | RunnableLambda(add_suffix)) # Applies the add_suffix function

# Invoke the pipeline
result = pipe.invoke({"text":"i wish i would've done my homework earlier"})

# Print the result
print(result)

## 6) Load a PDF and chunk it (preparation for RAG)

Retrieval Augmented Generation (RAG) involves retrieving relevant information from a knowledge base to help the LLM generate a response. The first step is often loading and processing documents.

We'll load a PDF and split it into smaller chunks.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
import requests


pdf_path = "3D_Systems_2025_Q2_Report.pdf"

print(f"Loading document from {pdf_path}")
# Load the PDF document
loader = PyPDFLoader(str(pdf_path))
docs = loader.load()

# Initialize the text splitter
# chunk_size: the maximum size of each chunk
# chunk_overlap: the number of characters to overlap between chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

# Split the documents into chunks
# Ensure docs list is not empty and documents have 'page_content' attribute
splits = splitter.split_documents(docs) if docs and hasattr(docs[0], "page_content") else []

print(f"Number of chunks: {len(splits)}")

# Print the beginning of the first chunk (or dummy text)
if splits:
    print("First chunk preview:")
    print(splits[0].page_content[:200].replace("\n", " "))
    print("\nSecond chunk preview")
    print(splits[1].page_content[:200].replace("\n", " "))
else:
    print("No splits available")

### Parameters for Document Processing & Embeddings

Before creating embeddings, we often process documents by splitting them into smaller chunks. The parameters for this text splitting process significantly impact the resulting embeddings and the performance of a RAG system.

Here are key parameters you can tune during the document chunking phase:

- **`chunk_size`**:
    - **Description:** The maximum number of characters in each document chunk.
    - **Importance:** This is crucial because language models and embedding models have input token limits. Smaller chunks help ensure that individual chunks don't exceed these limits and that the content within a chunk is topically cohesive.
    - **Guidelines:**
        - Too small: Chunks might lack sufficient context.
        - Too large: Chunks might exceed model limits or contain too much irrelevant information.
        - The optimal size depends on the nature of your documents and the specific embedding model and LLM you are using. Experimentation is key!

- **`chunk_overlap`**:
    - **Description:** The number of characters that overlap between consecutive chunks.
    - **Importance:** Overlap helps maintain context across chunk boundaries. If a critical piece of information is split between two chunks, overlap ensures that both parts are included in at least one chunk.
    - **Guidelines:**
        - Too little overlap: Important context might be lost between chunks.
        - Too much overlap: Can create redundant information and increase processing time and cost.
        - A common starting point is often around 10-20% of the `chunk_size`.

- **Text Splitter Strategy (e.g., `RecursiveCharacterTextSplitter`)**:
    - **Description:** The method used to split the text. Different splitters use different strategies (e.g., splitting by characters, words, sentences, paragraphs, or recursively trying different delimiters).
    - **Importance:** The splitting strategy determines how the text is broken down, which affects the content and context of each chunk. Choosing the right splitter for your document type (code, markdown, plain text, etc.) is important.
    - **Guidelines:** `RecursiveCharacterTextSplitter` is often a good default as it tries multiple delimiters, but specialized splitters exist for specific formats (like `MarkdownTextSplitter`, `PythonCodeTextSplitter`).

While the embedding *model* itself (like `text-embedding-3-small`) also has parameters (like the embedding dimension), tuning the text splitting parameters is often the first and most impactful step in preparing documents for embedding in a RAG pipeline.

## 7) Tiny RAG with FAISS

Now we'll create a simple RAG system. We'll use **FAISS** (Facebook AI Similarity Search) as a local vector store to index our document chunks and **OpenAI Embeddings** to convert text into vector representations.

**What is FAISS?**

FAISS is a library for efficient similarity search and clustering of dense vectors. It is developed by Facebook AI. It allows you to quickly search for the most similar vectors to a query vector within a large collection of vectors. This is exactly what we need for RAG: we convert our document chunks into vectors (embeddings), and then when a user asks a question, we convert the question into a vector and use FAISS to find the document chunks whose vectors are most similar to the question vector.

**How FAISS Functions:**

1.  **Indexing:** You add your vectors (document embeddings) to a FAISS index. FAISS uses various indexing structures and algorithms to organize these vectors for fast searching.
2.  **Searching:** When you have a query vector (the embedding of the user's question), you search the FAISS index. FAISS uses its optimized algorithms to quickly find the "nearest neighbors" – the vectors in the index that are most similar to your query vector based on metrics like cosine similarity.
3.  **Retrieval:** The indices of the most similar vectors are returned. These indices correspond to your original document chunks, which are then retrieved and passed to the language model as context.

In LangChain, the `FAISS.from_texts` function handles the indexing step, creating a FAISS index from your document chunks and their embeddings. The `.as_retriever()` method on the FAISS object then provides an interface to perform the searching and retrieval step when a query is given.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# Initialize OpenAI Embeddings model
# 'text-embedding-3-small' is a good balance of performance and cost
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

# Create a FAISS vector store from the document splits
if len(splits) > 0:
    print(f"Creating FAISS index from {len(splits)} document splits.")
    texts = [d.page_content for d in splits]
    vectorStore = FAISS.from_texts(texts, embedding=embeddings)

# Create a retriever from the vector store
# The retriever will fetch the most relevant documents based on a query
retriever = vectorStore.as_retriever(search_kwargs={"k":5}) # k=5 means retrieve the top 5 most similar documents

# Define a function to format the retrieved documents for the prompt
def format_docs(docs):
    """Formats a list of documents into a single string for the prompt."""
    return "\n\n".join(d.page_content[:300].replace("\n"," ") for d in docs) # Take first 300 chars, replace newlines

# Define the RAG prompt template
# This prompt instructs the LLM to answer based *only* on the provided context
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question based ONLY on the following context. If you don't know the answer, say you don't know."),
    ("human", "Question: {question}\n\nContext:\n{context}")
])

# Create the RAG chain
# This chain combines the retriever, the document formatter, the prompt, the LLM, and the output parser
rag_chain = (
    {"context": retriever | RunnableLambda(format_docs), # Retrieve and format documents
     "question": RunnablePassthrough()} # Pass the original question through
    | rag_prompt # Apply the RAG prompt
    | llm # Pass to the LLM
    | StrOutputParser() # Parse the output as a string
)

# Invoke the RAG chain with a question
print("\nInvoking RAG chain...")
response = rag_chain.invoke("What is LangChain useful for?")
response_2 = rag_chain.invoke("Tell me about 3D System's financial performance for Q2?")

# Print the response
print("\nAsking an unrelated question to the RAG bot:")
print(response)
print("\nAsking a question regarding the document to the RAG bot:")
print(response_2)

## 8) Tools + a minimal tool-calling agent

LangChain agents can use tools to interact with the outside world or perform specific actions. Here, we define two simple tools and create an agent that can decide when to use them.

In [ ]:
from langchain_core.tools import tool

# Define a tool for adding two numbers
def calc_add(a, b):
    "Add two numbers."
    return a + b

# Define a tool for our mini RAG system
def mini_rag(question):
    "Answer from the local FAISS-backed context."
    return rag_chain.invoke(question)

# List of available tools for the agent
tools = [calc_add, mini_rag]

print("Tools defined")

In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain.tools import tool
import json

# Initialize the ChatOpenAI model for the agent
# Using the same model and temperature as defined earlier (cell f6e414b9)
model = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

# Create the agent
# This function sets up the agent with the language model, tools, and a system prompt
agent = create_agent(
    model=model,
    tools=tools,
    system_prompt="You are a knowledgeable chatbot, use the available tools when necessary"
)

# Invoke the agent with a specific question
# The agent will decide if it needs to use any tools to answer the question
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "Tell me why 3D Systems is struggling"}
    ]
})

# Define a helper function to pretty print the agent's result
def pretty_agent_result(result):
    msgs = result["messages"]
    for i, m in enumerate(msgs, 1):
        # Determine the role (type) of the message
        role = getattr(m, "type", None) or getattr(m, "role", None) or f"msg{i}"
        content = getattr(m, "content", "")

        # Print the message number and role
        print(f"\n{i}. {role.upper()}")
        # Print the content, handling list format for tool calls
        if isinstance(content, list):
            print(json.dumps(content, indent=2, ensure_ascii=False))
        else:
            print(content or "")

        # Print additional kwargs, specifically tool calls
        ak = getattr(m, "additional_kwargs", {}) or {}
        if "tool_calls" in ak:
            print("tool_calls:")
            print(json.dumps(ak["tool_calls"], indent=2, ensure_ascii=False))

        # Print the tool name if available
        name = getattr(m, "name", None)
        if name:
            print(f"tool_name: {name}")

        # Print token usage if available in response metadata
        meta = getattr(m, "response_metadata", {}) or {}
        usage = meta.get("token_usage")
        if usage:
            print("token_usage:", {k: usage.get(k) for k in ("prompt_tokens", "completion_tokens", "total_tokens")})

# Use the helper function to display the result of the agent invocation
pretty_agent_result(result)

In [ ]:
# Invoke the agent with a question that requires the calculator tool
response = agent.invoke({"messages": [{"role": "user", "content": "Please add 12.5 and 7.25"}]})

print("\nAgent's final response:")
pretty_agent_result(response)

## Conclusion

This tutorial covered the fundamental concepts of LangChain, including calling LLMs, using prompts and output parsers, composing runnables, loading documents, building a simple RAG system, and creating a tool-calling agent.

You can now experiment with different models, prompts, documents, and tools to build more complex LLM applications!